# Alkahest 4B Two-Stage SFT

Runs the same Stage A + Stage B RP adapter recipe against `tvall43/Qwen3.5-4B-heretic`, producing `thomasjvu/alkahest-4b-heretic-rp-merged` for text-only q4 ONNX export.

In [ ]:
import os, platform, shutil, subprocess, sys, time
from pathlib import Path

MODEL_NAME = os.environ.get('MODEL_NAME', 'tvall43/Qwen3.5-4B-heretic')
MERGED_UPLOAD_REPO = os.environ.get('MERGED_UPLOAD_REPO', 'thomasjvu/alkahest-4b-heretic-rp-merged')
WORK_DIR = Path(os.environ.get('TWO_STAGE_WORK_DIR', '/kaggle/working/alkahest-4b-two-stage-sft'))

print('python_platform=', platform.platform())
print('working_disk_free_gb=', round(shutil.disk_usage('/kaggle/working').free / 1024**3, 2))
print('model_name=', MODEL_NAME)
print('merged_upload_repo=', MERGED_UPLOAD_REPO)
try:
    import torch
    print('torch=', torch.__version__)
    print('cuda_available=', torch.cuda.is_available())
    print('gpu_count=', torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        major, minor = torch.cuda.get_device_capability(i)
        print(f'gpu_{i}=', props.name, round(props.total_memory / 1024**3, 2), 'GiB', f'sm_{major}{minor}')
    if torch.cuda.is_available():
        major, minor = torch.cuda.get_device_capability(0)
        if major < 7:
            raise RuntimeError('Kaggle assigned a pre-Volta GPU. Push with --accelerator NvidiaTeslaT4.')
except Exception as exc:
    print('torch_probe_error=', repr(exc))
    if 'pre-Volta' in str(exc):
        raise

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
secret_token = ''
for attempt in range(5):
    try:
        from kaggle_secrets import UserSecretsClient
        secret_token = UserSecretsClient().get_secret('HF_TOKEN')
        break
    except Exception as exc:
        print('hf_secret_attempt_failed=', attempt + 1, type(exc).__name__)
        time.sleep(3)
if secret_token and not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = secret_token
if secret_token and not os.environ.get('HUGGING_FACE_HUB_TOKEN'):
    os.environ['HUGGING_FACE_HUB_TOKEN'] = secret_token
print('hf_secret_loaded=', bool(secret_token))

packages = [
    'git+https://github.com/huggingface/transformers.git@c472755e79aac54d675845bff5e5c821c21260af',
    'accelerate>=1.13.0',
    'bitsandbytes>=0.49.0',
    'peft>=0.19.0',
    'datasets>=4.8.0',
    'huggingface_hub[cli]>=1.5.0',
    'hf_transfer>=0.1.9',
    'safetensors>=0.7.0',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])
subprocess.call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'])
token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')
if token:
    try:
        from huggingface_hub import HfApi
        print('hf_whoami=', HfApi(token=token).whoami().get('name', 'unknown'))
    except Exception as exc:
        print('hf_token_invalid=', type(exc).__name__)
        os.environ.pop('HF_TOKEN', None)
        os.environ.pop('HUGGING_FACE_HUB_TOKEN', None)
print('hf_token_present=', bool(os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')))


In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = os.environ.get('HERETIC_TO_ONNX_REPO', 'https://github.com/alkahest-ai/heretic-to-onnx.git')
REPO_REF = os.environ.get('HERETIC_TO_ONNX_REF', 'codex/kaggle-heretic-2b-run')
REPO_DIR = Path('/kaggle/working/heretic-to-onnx')
if REPO_DIR.exists():
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_REF])
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'checkout', REPO_REF])
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'])
else:
    subprocess.check_call(['git', 'clone', '--branch', REPO_REF, '--depth', '1', REPO_URL, str(REPO_DIR)])
print('repo=', REPO_DIR)
print('head=', subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip())

WORK_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_DIR = WORK_DIR / 'splits'
subprocess.check_call([
    sys.executable,
    str(REPO_DIR / 'scripts/prepare_alkahest_two_stage_sft.py'),
    '--output-dir', str(SPLIT_DIR),
    '--stage-a-repeats', os.environ.get('TWO_STAGE_A_REPEATS', '18'),
    '--stage-b-boundary-repeats', os.environ.get('TWO_STAGE_B_BOUNDARY_REPEATS', '2'),
    '--stage-b-adult-repeats', os.environ.get('TWO_STAGE_B_ADULT_REPEATS', '56'),
    '--val-fraction', os.environ.get('TWO_STAGE_VAL_FRACTION', '0.10'),
])

def run_train(stage, model_name, train_file, val_file, output_dir, merged_dir, defaults):
    cmd = [
        sys.executable,
        str(REPO_DIR / 'scripts/train_alkahest_sft.py'),
        '--model-name', str(model_name),
        '--train-file', str(train_file),
        '--val-file', str(val_file),
        '--dataset-manifest', str(SPLIT_DIR / 'manifest.json'),
        '--output-dir', str(output_dir),
        '--merged-output-dir', str(merged_dir),
        '--max-seq-length', os.environ.get(f'{stage}_MAX_SEQ_LENGTH', defaults['seq']),
        '--max-steps', os.environ.get(f'{stage}_MAX_STEPS', defaults['steps']),
        '--gradient-accumulation-steps', os.environ.get(f'{stage}_GRAD_ACCUM', '8'),
        '--learning-rate', os.environ.get(f'{stage}_LR', defaults['lr']),
        '--eval-steps', os.environ.get(f'{stage}_EVAL_STEPS', defaults['eval']),
        '--save-steps', os.environ.get(f'{stage}_SAVE_STEPS', defaults['save']),
    ]
    if defaults.get('upload') and os.environ.get('SFT_NO_UPLOAD') != '1' and (os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')):
        cmd.extend(['--upload-merged-to', MERGED_UPLOAD_REPO])
    print('running=', ' '.join(cmd))
    env = os.environ.copy()
    env['CUDA_VISIBLE_DEVICES'] = os.environ.get('CUDA_VISIBLE_DEVICES', '0')
    subprocess.check_call(cmd, env=env)

run_train('STAGE_A', MODEL_NAME, SPLIT_DIR / 'stage_a' / 'train.jsonl', SPLIT_DIR / 'stage_a' / 'val.jsonl', WORK_DIR / 'stage-a-adapter', WORK_DIR / 'stage-a-merged', {'seq': '512', 'steps': '60', 'lr': '3e-6', 'eval': '15', 'save': '30'})
run_train('STAGE_B', WORK_DIR / 'stage-a-merged', SPLIT_DIR / 'stage_b' / 'train.jsonl', SPLIT_DIR / 'stage_b' / 'val.jsonl', WORK_DIR / 'stage-b-adapter', WORK_DIR / 'stage-ab-merged', {'seq': '512', 'steps': '50', 'lr': '2e-6', 'eval': '10', 'save': '25', 'upload': True})


In [ ]:
from pathlib import Path
import json, os

for report in [WORK_DIR / 'stage-a-adapter' / 'training-run.json', WORK_DIR / 'stage-b-adapter' / 'training-run.json']:
    if report.exists():
        data = json.loads(report.read_text())
        print(report)
        print(json.dumps({
            'ok': data.get('ok'),
            'model_name': data.get('model_name'),
            'train_rows': data.get('train_rows'),
            'val_rows': data.get('val_rows'),
            'max_steps': data.get('max_steps'),
            'merged': data.get('merged'),
            'uploaded_merged_to': data.get('uploaded_merged_to'),
            'upload_results': data.get('upload_results'),
        }, indent=2))
